# 03_01_behavior_EDA

This notebook performs EDA for the behavior variable **SadOrHopeless**.
All major tables and figures include short explanations.

## Step 1: Load data
Read the processed dataset, define paths, and prepare output folders.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Paths
input_path = "../data/processed/cycle2_vddc_imputed.csv"
missing_summary_path = "../outputs/tables/missing_summary.csv"

tables_dir = "../outputs/tables"
figures_dir = "../outputs/figures"

os.makedirs(tables_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)

# Variable
behavior_var = "SadOrHopeless"

# Load data
df = pd.read_csv(input_path)

print("Dataset shape:", df.shape)
print("Columns loaded:", list(df.columns))

簡短說明：  
This step loads the processed data and prepares the folders for saving EDA outputs.

## Step 2: Confirm behavior variable
Check that the required behavior variable exists and define the coding rule.

In [ ]:
if behavior_var not in df.columns:
    raise ValueError(f"{behavior_var} is not found in the dataset.")

print("Behavior variable:", behavior_var)
print("Coding rule:")
print("  Success = 1")
print("  Failure = 2")

簡短說明：  
The selected behavior variable is **SadOrHopeless**. In this project, code **1** is treated as success and code **2** is treated as failure.

## Step 3: Table of original code frequencies
Create the original code frequency table, the binary recoded table, and the success/failure summary table.
Also display and save bar charts for the original codes and the final binary categories.

In [ ]:
# Original code frequency table
original_freq = (
    df[behavior_var]
    .value_counts(dropna=False)
    .sort_index()
    .reset_index()
)
original_freq.columns = ["code", "count"]

# Save original code frequency table
original_freq_path = os.path.join(tables_dir, "behavior_original_code_frequencies.csv")
original_freq.to_csv(original_freq_path, index=False)

print("Original code frequency table:")
display(original_freq)
print(f"Saved to: {original_freq_path}")

簡短說明：  
This table shows how often each original code appears in **SadOrHopeless**. It helps us understand the raw distribution before binary interpretation.

In [ ]:
# Recode to labeled binary categories
binary_labeled = df[behavior_var].map({1: "Success", 2: "Failure"})

binary_freq = (
    binary_labeled
    .value_counts(dropna=False)
    .reindex(["Success", "Failure"])
    .fillna(0)
    .astype(int)
    .reset_index()
)
binary_freq.columns = ["category", "count"]

# Save binary recoded frequency table
binary_freq_path = os.path.join(tables_dir, "behavior_binary_recoded_frequencies.csv")
binary_freq.to_csv(binary_freq_path, index=False)

print("Binary recoded frequency table:")
display(binary_freq)
print(f"Saved to: {binary_freq_path}")

簡短說明：  
After recoding, the behavior variable is reduced to two categories: **Success** and **Failure**. This is the form used for the proportion analysis.

In [ ]:
# Success and failure counts + proportions
sf_table = binary_freq.copy()
total_valid = sf_table["count"].sum()
sf_table["proportion"] = sf_table["count"] / total_valid

sf_table_path = os.path.join(tables_dir, "behavior_success_failure_summary.csv")
sf_table.to_csv(sf_table_path, index=False)

print("Success and failure summary table:")
display(sf_table)
print(f"Saved to: {sf_table_path}")

簡短說明：  
This table reports both the counts and proportions of success and failure. The proportion of success is especially important for the later one-sample proportion inference.

In [ ]:
# Bar chart: original code frequencies
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(original_freq["code"].astype(str), original_freq["count"])
ax.set_title("SadOrHopeless: Original Code Frequencies")
ax.set_xlabel("Original Code")
ax.set_ylabel("Count")

original_fig_path = os.path.join(figures_dir, "behavior_original_code_frequencies.png")
fig.savefig(original_fig_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Saved figure to: {original_fig_path}")

簡短說明：  
This bar chart visualizes the original code distribution of **SadOrHopeless**. It provides a quick visual check of how the raw responses are distributed.

In [ ]:
# Bar chart: success and failure
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(sf_table["category"], sf_table["count"])
ax.set_title("SadOrHopeless: Success and Failure")
ax.set_xlabel("Category")
ax.set_ylabel("Count")

sf_fig_path = os.path.join(figures_dir, "behavior_success_failure.png")
fig.savefig(sf_fig_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Saved figure to: {sf_fig_path}")

簡短說明：  
This bar chart compares the number of success and failure observations after binary recoding. It directly supports the behavior-variable proportion analysis.

## Step 4: Missing / invalid value count
Read `../outputs/tables/missing_summary.csv`, filter the behavior variable, and display a bar chart of valid, missing, and invalid counts.

In [ ]:
missing_summary = pd.read_csv(missing_summary_path)

behavior_missing = missing_summary.loc[
    missing_summary["variable"] == behavior_var
].copy()

if behavior_missing.empty:
    raise ValueError(f"No row for {behavior_var} found in missing_summary.csv")

display(behavior_missing)

簡短說明：  
This row shows the data quality summary for the selected behavior variable, including valid, missing, and invalid counts.

In [ ]:
row = behavior_missing.iloc[0]

quality_df = pd.DataFrame({
    "status": ["valid", "missing", "invalid"],
    "count": [row["valid_count"], row["missing_count"], row["invalid_count"]]
})

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(quality_df["status"], quality_df["count"])
ax.set_title("SadOrHopeless: Valid, Missing, and Invalid Counts")
ax.set_xlabel("Status")
ax.set_ylabel("Count")

quality_fig_path = os.path.join(figures_dir, "behavior_missing_invalid_valid.png")
fig.savefig(quality_fig_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Saved figure to: {quality_fig_path}")

簡短說明：  
This chart shows the number of valid, missing, and invalid observations for **SadOrHopeless**. It helps document data quality before inference.